# The position battery against Cloudict

Fourteen fixtures selected from `benchmarks/battery.json`, played by the compiled
engine against Cloudict: fresh engine processes per fixture, identical injected
history, a 3-second end-to-end reply deadline and a 10-minute run cap.

In [1]:
import hashlib
import json
import os
import queue
import re
import shutil
import subprocess
import sys
import tempfile
import threading
import time
from pathlib import Path

sys.dont_write_bytecode = True  # never write __pycache__ into the existing project tree


def find_root():
    for base in [Path.cwd(), *Path.cwd().parents]:
        if (base / "connect6" / "defines.py").is_file():
            return base.resolve()
    raise RuntimeError("run this notebook from inside the Connect6Engine repository")


ROOT = find_root()
sys.path.insert(0, str(ROOT))
sys.path.insert(0, str(ROOT / "tuning"))
import harness  # reuse: replay_history, battery_horizon_plies (read-only)

from connect6.board import check_game_end, make_move, opponent
from connect6.defines import Color, GameResult
from connect6.protocol import msg2move

HERE = ROOT / "benchmarks" / "assignment_coverage"
RECORDS = HERE / "records"
LOGS = HERE / "logs"
RECORDS.mkdir(parents=True, exist_ok=True)
LOGS.mkdir(parents=True, exist_ok=True)
OUT_JSON = RECORDS / "01_cloudict_positions.json"

OUR_BIN = ROOT / "dist" / "Connect6_Butovka_Ahmed"
CLOUDICT_BIN = ROOT / "benchmarks" / "cloudict" / "gameEngine"
PATTERNS = ROOT / "benchmarks" / "cloudict" / "patterns.in"
for path in (OUR_BIN, CLOUDICT_BIN, PATTERNS):
    assert path.is_file(), f"missing prerequisite, experiment blocked: {path}"

FLOOR = 2                 # our engine: depth command = budget-mode floor
CLOUDICT_DEPTH = 3        # cloudict: fixed-depth search (plus its own VCF)
REPLY_DEADLINE = 3.0      # end-to-end reply deadline, seconds
RUN_LIMIT = 600.0         # whole-notebook run cap, seconds
OUR_ENV = dict(os.environ, V6_JOBS="1", V6_TIME_BUDGET="2", PYTHONDONTWRITEBYTECODE="1")

SELECTED = [
    "win-1-horizontal-live-five", "win-1-diagonal-live-five",
    "win-2-seedJJ", "win-2-seedKJIJ",
    "win-3-seedJJ", "win-3-seedHIKK",
    "defend-1-horizontal-live-five", "defend-1-diagonal-live-five",
    "defend-2-early-fork", "defend-2-double-threat",
    "defend-3-opening-fork", "defend-3-fork-build",
    "start-empty-black-survives-opening", "start-empty-white-converts-opening",
]

battery = json.loads((ROOT / "benchmarks" / "battery.json").read_text())
by_id = {entry["id"]: entry for entry in battery}
fixtures = [by_id[fid] for fid in SELECTED]
assert len(fixtures) == 14 and all(fixtures), "fixture selection must resolve in battery.json"


def sha12(path):
    return hashlib.sha256(Path(path).read_bytes()).hexdigest()[:12]


CONFIG = {
    "ours": {
        "engine": "compiled V6 (dist build)",
        "bin": str(OUR_BIN.relative_to(ROOT)),
        "sha256_12": sha12(OUR_BIN),
        "weights": "tuned champion (launcher-pinned dist/lib/tuning/winner.json)",
        "book": "existing opening book (launcher-pinned dist/lib/tuning/opening_book.json)",
        "floor_depth": FLOOR,
        "V6_JOBS": 1,
        "V6_TIME_BUDGET_s": 2.0,
    },
    "cloudict": {
        "bin": str(CLOUDICT_BIN.relative_to(ROOT)),
        "sha256_12": sha12(CLOUDICT_BIN),
        "depth": CLOUDICT_DEPTH,
        "forced_win_search": "normal (always-on VCF), no protocol command sent",
    },
    "reply_deadline_s": REPLY_DEADLINE,
    "run_limit_s": RUN_LIMIT,
    "retries": "none - one deterministic attempt per fixture",
    "reached_depth": "unavailable: the text protocol of neither engine prints a reached-depth line",
    "python": sys.version.split()[0],
    "platform": sys.platform,
}

# pre-flight: every selected history replays legally and the side to move matches
print(f"{'id':<38} {'class':<8} {'us':<6} {'hist':>4} {'opens':<7} mover after history")
for entry in fixtures:
    board, _pre, mover = harness.replay_history(entry["history"])
    mover_name = {1: "black", 2: "white"}[int(mover)]
    expected = entry["us"] if entry["class"] in ("win", "defend") else mover_name
    assert mover_name == expected, f"{entry['id']}: mover {mover_name} != expected {expected}"
    print(f"{entry['id']:<38} {entry['class']:<8} {entry['us']:<6} {len(entry['history']):>4} "
          f"{entry['history'][0] if entry['history'] else '-':<7} {mover_name}")
print("\nall 14 histories replay legally; sides to move match the fixtures")

id                                     class    us     hist opens   mover after history
win-1-horizontal-live-five             win      black     6 JJ      black
win-1-diagonal-live-five               win      black     6 DP      black
win-2-seedJJ                           win      white    17 JJ      white
win-2-seedKJIJ                         win      white    11 JJ      white
win-3-seedJJ                           win      white    15 JJ      white
win-3-seedHIKK                         win      white    35 JJ      white
defend-1-horizontal-live-five          defend   white     5 JJ      white
defend-1-diagonal-live-five            defend   white     5 DP      white
defend-2-early-fork                    defend   black     8 JJ      black
defend-2-double-threat                 defend   black    22 JJ      black
defend-3-opening-fork                  defend   black     2 JJ      black
defend-3-fork-build                    defend   black    12 JJ      black
start-empty-black-surviv

In [2]:
MOVE_RE = re.compile(r"^move ([A-S]{2,4})\s*$")
NODE_RE = re.compile(r"^Node\s*:\s*(\d+)")
SCORE_RE = re.compile(r"^Score\s*:\s*(-?[\d.]+)")
SEARCH_TIME_RE = re.compile(r"^Search Time:\s*([\d.]+)")


class Engine:
    # Text-protocol wrapper: a background reader queues output lines, replies are read with a hard timeout.

    def __init__(self, proc, tag):
        self.proc = proc
        self.tag = tag
        self.stdin = proc.stdin
        self.stdout = proc.stdout
        self.lines = queue.Queue()
        threading.Thread(target=self._reader, daemon=True).start()
        self.cmd("name")  # identification reply is consumed lazily with the preamble

    def _reader(self):
        for line in self.stdout:
            self.lines.put(line.rstrip("\n"))

    def cmd(self, line):
        self.stdin.write(line + "\n")
        self.stdin.flush()

    def read_reply(self, timeout):
        lines, move = [], None
        while move is None:
            try:
                line = self.lines.get(timeout=timeout)
            except queue.Empty:
                raise TimeoutError(f"{self.tag}: no reply within {timeout} s") from None
            lines.append(line)
            matched = MOVE_RE.match(line)
            if matched:
                move = matched.group(1)
        return move, lines

    def close(self):
        try:
            self.cmd("exit")
            self.stdin.close()
            self.proc.wait(timeout=10)
        except Exception:
            try:
                self.proc.kill()  # targeted kill: never the notebook's own process group
                self.proc.wait(timeout=10)
            except Exception:
                pass


def stats_from(lines):
    stats = {}
    for line in lines:
        matched = NODE_RE.match(line)
        if matched:
            stats["nodes"] = int(matched.group(1))
        matched = SCORE_RE.match(line)
        if matched:
            stats["score"] = float(matched.group(1))
        matched = SEARCH_TIME_RE.match(line)
        if matched:
            stats["search_s"] = float(matched.group(1))
        if line.startswith("Book:"):
            stats["book"] = True
    return stats


def color_name(color):
    return "black" if color is Color.BLACK else "white"


def run_fixture(entry, index, workroot):
    us = Color.BLACK if entry["us"] == "black" else Color.WHITE
    them = opponent(us)
    horizon = harness.battery_horizon_plies(entry)
    history = list(entry["history"])
    board, _pre, mover = harness.replay_history(history)
    if entry["class"] in ("win", "defend"):
        assert mover is us, f"{entry['id']}: side to move after history is {color_name(mover)}"

    workdir = Path(tempfile.mkdtemp(prefix=f"fix{index:02d}_", dir=str(workroot)))
    shutil.copy(PATTERNS, workdir)  # cloudict reads patterns.in from its cwd
    ours_err_path = LOGS / f"{entry['id']}_ours_stderr.txt"
    cloud_err_path = LOGS / f"{entry['id']}_cloudict_stderr.txt"
    ours_err = open(ours_err_path, "w")
    cloud_err = open(cloud_err_path, "w")
    ours = cloudict = None
    moves = []
    winner = None  # Color | "draw" | None
    plies = 0
    status = None
    timeout_at = None
    error = None

    def spawn_ours():
        proc = subprocess.Popen(
            [str(OUR_BIN)], stdin=subprocess.PIPE, stdout=subprocess.PIPE,
            stderr=ours_err, text=True, bufsize=1, cwd=str(workdir), env=OUR_ENV)
        return Engine(proc, "ours")

    def spawn_cloudict():
        proc = subprocess.Popen(
            [str(CLOUDICT_BIN)], stdin=subprocess.PIPE, stdout=subprocess.PIPE,
            stderr=cloud_err, text=True, bufsize=1, cwd=str(workdir))
        return Engine(proc, "cloudict")

    def apply_and_record(engine, reply, color, wall, lines):
        nonlocal plies, winner
        move = msg2move(reply)
        make_move(board, move, color)
        plies += 1
        rec = {"ply": plies, "color": color_name(color), "engine": engine.tag,
               "msg": reply, "wall_s": round(wall, 3)}
        rec.update(stats_from(lines))
        moves.append(rec)
        end = check_game_end(board, move)
        if end == GameResult.WIN:
            winner = color
        elif end == GameResult.DRAW:
            winner = "draw"

    def inject(engine, msgs, first_color):
        color = first_color
        for msg in msgs:
            engine.cmd(f"{color_name(color)} {msg}")
            color = opponent(color)

    try:
        ours = spawn_ours()
        cloudict = spawn_cloudict()
        ours.cmd(f"depth {FLOOR}")
        cloudict.cmd(f"depth {CLOUDICT_DEPTH}")

        # ---- identical setup in both engines (recipes proven against both binaries) ----
        # A black side that opens at the centre: 'new black' plays and announces JJ, so the
        # reply is the fixture's own first stone. Off-centre openings (the diagonal twins
        # open at DP) cannot use 'new black' - both engines force JJ there - so the whole
        # history is injected as plain placements after 'new white'; natural order ends on
        # the true last stone, and 'next'/'move' then search for the correct side.
        if us is Color.BLACK and (not history or history[0] == "JJ"):
            ours.cmd("new black")
            t0 = time.perf_counter()
            reply, lines = ours.read_reply(REPLY_DEADLINE)
            wall = time.perf_counter() - t0
            if history:
                assert reply == history[0] == "JJ", f"our engine must open JJ, got {reply}"
            else:
                apply_and_record(ours, reply, us, wall, lines)  # start-empty-black: our ply 1
            inject(ours, history[1:], Color.WHITE)
        else:
            ours.cmd("new white")
            inject(ours, history, Color.BLACK)

        if them is Color.BLACK and (not history or history[0] == "JJ"):
            cloudict.cmd("new black")
            t0 = time.perf_counter()
            reply, lines = cloudict.read_reply(REPLY_DEADLINE)
            wall = time.perf_counter() - t0
            if history:
                assert reply == history[0] == "JJ", f"cloudict must open JJ, got {reply}"
            else:
                apply_and_record(cloudict, reply, them, wall, lines)  # their ply 1
            inject(cloudict, history[1:], Color.WHITE)
        else:
            cloudict.cmd("new white")
            inject(cloudict, history, Color.BLACK)

        # ---- our first move (we are the side to move at the fixture position) ----
        if history:
            ours.cmd("next")  # toggle to our colour and search
            t0 = time.perf_counter()
            reply, lines = ours.read_reply(REPLY_DEADLINE)
            apply_and_record(ours, reply, us, time.perf_counter() - t0, lines)
        elif us is Color.WHITE:
            # start-empty-white: cloudict opened during setup; relay its move
            opening = moves[0]["msg"]
            ours.cmd(f"move {opening}")
            t0 = time.perf_counter()
            reply, lines = ours.read_reply(REPLY_DEADLINE)
            apply_and_record(ours, reply, us, time.perf_counter() - t0, lines)

        # ---- alternate until horizon or terminal result ----
        next_is_us = False
        while winner is None and plies < horizon:
            engine = ours if next_is_us else cloudict
            color = us if next_is_us else them
            last = moves[-1]
            engine.cmd(f"move {last['msg']}")
            t0 = time.perf_counter()
            reply, lines = engine.read_reply(REPLY_DEADLINE)
            apply_and_record(engine, reply, color, time.perf_counter() - t0, lines)
            next_is_us = not next_is_us
    except TimeoutError as exc:
        timeout_at = plies + 1
        status = "timeout"
        error = str(exc)
    except Exception as exc:  # record honestly, never repair mid-run
        status = "incomplete"
        error = repr(exc)
    finally:
        for engine in (ours, cloudict):
            if engine is not None:
                engine.close()
        ours_err.close()
        cloud_err.close()

    if status is None:
        if entry["class"] == "win":
            passed = winner is us
        elif entry["class"] == "defend":
            passed = winner is not them
        else:  # start: a loss exactly on ply 20 is not "before" ply 20
            passed = not (winner is them and plies < horizon)
        status = "pass" if passed else "fail"

    def tally(tag):
        replies = [m for m in moves if m["engine"] == tag]
        walls = [m["wall_s"] for m in replies]
        nodes = [m["nodes"] for m in replies if "nodes" in m]
        searches = [m["search_s"] for m in replies if "search_s" in m]
        return {
            "replies": len(replies),
            "wall_mean_s": round(sum(walls) / len(walls), 3) if walls else None,
            "wall_max_s": max(walls) if walls else None,
            "search_max_s": max(searches) if searches else None,
            "nodes_total": sum(nodes) if nodes else None,
            "book_hits": sum(1 for m in replies if m.get("book")),
        }

    return {
        "id": entry["id"],
        "class": entry["class"],
        "turns": entry.get("turns"),
        "min_plies": entry.get("min_plies"),
        "us": entry["us"],
        "history": history,
        "horizon_plies": horizon,
        "status": status,
        "winner": None if winner is None else ("draw" if winner == "draw" else color_name(winner)),
        "plies": plies,
        "moves": moves,
        "ours": tally("ours"),
        "cloudict": tally("cloudict"),
        "timeout_at_ply": timeout_at,
        "reached_depth_available": False,
        "stderr_bytes": {"ours": ours_err_path.stat().st_size, "cloudict": cloud_err_path.stat().st_size},
        "battery_note": entry.get("note", ""),
        "error": error,
    }


print("runner ready: fresh engines per fixture, 3 s reply deadline, no retries")

runner ready: fresh engines per fixture, 3 s reply deadline, no retries


In [3]:
records = []
workroot = Path(tempfile.mkdtemp(prefix="assign_cloudict_"))
run_start = time.perf_counter()
for index, entry in enumerate(fixtures):
    elapsed = time.perf_counter() - run_start
    if elapsed > RUN_LIMIT - 30:
        # not enough headroom left in the 10-minute budget to start another fixture
        record = {"id": entry["id"], "status": "incomplete",
                  "error": "10-minute run cap reached before this fixture started"}
        print(f"[{index + 1:2d}/14] {entry['id']:<38} INCOMPLETE (run cap)", flush=True)
    else:
        try:
            record = run_fixture(entry, index, workroot)
        except Exception as exc:
            record = {"id": entry["id"], "status": "incomplete", "error": repr(exc)}
        print(f"[{index + 1:2d}/14] {entry['id']:<38} {record['status']:<10} "
              f"plies={record.get('plies', 0):<3} winner={record.get('winner')}", flush=True)
    records.append(record)
    OUT_JSON.write_text(json.dumps({"meta": CONFIG, "fixtures": records}, indent=1))  # crash-safe
run_wall = time.perf_counter() - run_start
shutil.rmtree(workroot, ignore_errors=True)
print(f"\nrun wall: {run_wall:.1f}s of the {RUN_LIMIT:.0f}s limit")

[ 1/14] win-1-horizontal-live-five             pass       plies=1   winner=black


[ 2/14] win-1-diagonal-live-five               pass       plies=1   winner=black


[ 3/14] win-2-seedJJ                           timeout    plies=0   winner=None


[ 4/14] win-2-seedKJIJ                         pass       plies=3   winner=white


[ 5/14] win-3-seedJJ                           fail       plies=5   winner=None


[ 6/14] win-3-seedHIKK                         timeout    plies=2   winner=None


... (5 outputs left out)

[12/14] defend-3-fork-build                    timeout    plies=0   winner=None


[13/14] start-empty-black-survives-opening     fail       plies=12  winner=white


[14/14] start-empty-white-converts-opening     fail       plies=11  winner=black



run wall: 48.8s of the 600s limit


In [4]:
print(f"{'fixture':<34} {'class':<10} {'status':<10} {'winner':<7} {'plies':>5} "
      f"{'hor':>4} {'ours nodes':>11} {'ours s/mv':>9} {'cloud s/mv':>10}")
for r in records:
    ours_t = r.get("ours") or {}
    cloud_t = r.get("cloudict") or {}
    print(f"{r['id']:<34} {r.get('class', '-'):<10} {r['status']:<10} {str(r.get('winner')):<7} "
          f"{r.get('plies', 0):>5} {r.get('horizon_plies', 0):>4} "
          f"{str(ours_t.get('nodes_total')):>11} {str(ours_t.get('wall_mean_s')):>9} "
          f"{str(cloud_t.get('wall_mean_s')):>10}")

from collections import Counter

status_counts = Counter(r["status"] for r in records)
print(f"\nstatus counts: {dict(status_counts)} of {len(records)}")
by_class = {}
for r in records:
    key = r.get("class", "?")
    by_class.setdefault(key, []).append(r["status"])
for cls, statuses in by_class.items():
    print(f"  {cls:<7}: {dict(Counter(statuses))}")

deadline_misses = [r["id"] for r in records if r["status"] == "timeout"]
target_misses = [m["msg"] for r in records for m in r.get("moves", [])
                 if m["engine"] == "ours" and m["wall_s"] > 2.0]
print(f"3 s deadline misses (fixtures stopped): {deadline_misses or 'none'}")
print(f"our replies over the 2 s budget target: {len(target_misses)}")

fixture                            class      status     winner  plies  hor  ours nodes ours s/mv cloud s/mv
win-1-horizontal-live-five         win        pass       black       1    1           1     0.001       None
win-1-diagonal-live-five           win        pass       black       1    1           1     0.075       None
win-2-seedJJ                       win        timeout    None        0    3        None      None       None
win-2-seedKJIJ                     win        pass       white       3    3           3      0.03      0.024
win-3-seedJJ                       win        fail       None        5    5        9081     1.388      0.035
win-3-seedHIKK                     win        timeout    None        2    5        3949     1.083      0.153
defend-1-horizontal-live-five      defend     pass       None        2    2        2353     0.459      0.149
defend-1-diagonal-live-five        defend     pass       None        2    2        2013     0.389      0.133
defend-2-early-fork

In [5]:
import io
import contextlib
from connect6.protocol import print_board


def render(board):
    buf = io.StringIO()
    with contextlib.redirect_stdout(buf):
        print_board(board)  # reuse the engine's own board printer, read-only
    return buf.getvalue()


def final_board(entry, record):
    full = list(entry["history"]) + [m["msg"] for m in record.get("moves", [])]
    board, _pre, _mover = harness.replay_history(full)
    return board


def explain(entry, record):
    lines = [f"{record['id']} — {record['status']} ({entry['class']}"
             + (f", {entry['turns']} turns" if entry.get("turns") else ", opening") + ")"]
    moves = record.get("moves", [])
    if not moves:
        lines.append("no moves recorded: " + str(record.get("error")))
        return "\n".join(lines)
    if record.get("winner") == entry["us"]:
        win_move = moves[-1]
        lines.append(f"we ({entry['us']}) won on ply {win_move['ply']} with {win_move['msg']} "
                     f"after {win_move.get('search_s', 0)} s of search ({win_move.get('nodes')} nodes); "
                     f"the horizon was {record['horizon_plies']} plies.")
    elif record.get("winner") == "draw":
        lines.append(f"draw at ply {moves[-1]['ply']}.")
    elif record.get("winner"):
        loss_move = moves[-1]
        our_scores = [m.get("score") for m in moves if m["engine"] == "ours" and "score" in m]
        seen = ("our last recorded score before the end was "
                + f"{our_scores[-1]:.0f}" if our_scores else "we never scored a reply")
        lines.append(f"{record['winner']} won on ply {loss_move['ply']} with {loss_move['msg']}; {seen}.")
    else:
        lines.append(f"no result within the {record['horizon_plies']}-ply horizon "
                     f"({len(moves)} plies played).")
    return "\n".join(lines)


passed = [r for r in records if r["status"] == "pass"]
not_passed = [r for r in records if r["status"] in ("fail", "timeout", "incomplete")]

for label, chosen in (("a success", passed[:1]), ("a failure", not_passed[:1])):
    for record in chosen:
        entry = by_id[record["id"]]
        print("=" * 46)
        print(explain(entry, record))
        print(render(final_board(entry, record)))
if not passed:
    print("no fixture passed - nothing to show as a success board")
if not not_passed:
    print("no failing fixture - nothing to show as a failure board")

win-1-horizontal-live-five — pass (win, 1 turns)
we (black) won on ply 1 with JOJI after 0.001 s of search (1 nodes); the horizon was 1 plies.
   A B C D E F G H I J K L M N O P Q R S 
A  - - - - - - - - - - - - - - - - - - - A
B  - - - - - - - - - - - - - - - - - - - B
C  - - - - - - - - - - - - - - - - - - - C
D  - - - - - - - - - - - - - - * * - - - D
E  - - - - - - - - - - - - - - * * - - - E
F  - - - - - - - - - - - - - - * * - - - F
G  - - - - - - - - - - - - - - - - - - - G
H  - - - - - - - - - - - - - - - - - - - H
I  - - - - - - - - - - - - - - - - - - - I
J  - - - - - - - - O O O O O O O - - - - J
K  - - - - - - - - - - - - - - - - - - - K
L  - - - - - - - - - - - - - - - - - - - L
M  - - - - - - - - - - - - - - - - - - - M
N  - - - - - - - - - - - - - - - - - - - N
O  - - - - - - - - - - - - - - - - - - - O
P  - - - - - - - - - - - - - - - - - - - P
Q  - - - - - - - - - - - - - - - - - - - Q
R  - - - - - - - - - - - - - - - - - - - R
S  - - - - - - - - - - - - - - - - - - - 

## Notes

Per-fixture records are saved incrementally to
`benchmarks/assignment_coverage/records/01_cloudict_positions.json`.

Reproduction: `uv run python scripts/nbclient_run.py benchmarks/assignment_coverage/01_cloudict_positions.ipynb`